In [1]:
import tensorflow as tf
import numpy as np
import gym
from ou_noise import OUNoise
import random


LAYER_1 = 400
LAYER_2 = 300
LAYER_3 = 300
keep_rate = 0.8
LAMBDA = 0.00001 # regularization term
GAMMA = 0.99
class DDPG(object):


    def __init__(self, sess, state_dim, action_dim, max_action, min_action, actor_learning_rate, critic_learning_rate, tau, RANDOM_SEED, device = '/cpu:0'):

        self.sess = sess
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        self.s_dim = state_dim
        self.a_dim = action_dim
        self.actor_learning_rate = actor_learning_rate
        self.critic_learning_rate = critic_learning_rate
        self.tau = tau
        self.device = device
        self.max_action = max_action
        self.min_action = min_action
        # Placeholders
        self.inputs = tf.placeholder(tf.float32, shape=[None, self.s_dim], name='state')
        self.action = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='actions')
        scope = 'net'    
        self.v, self.a, self.scaled_a, self.saver = self._build_net(scope)
        self.a_params = tf.trainable_variables(scope=scope + '/actor')
        self.c_params = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')
        scope = 'target'    
        self.v_target, self.a_target, self.scaled_a_target, self.saver_target = self._build_net(scope)
        self.a_params_target = tf.trainable_variables(scope=scope + '/actor')
        self.c_params_target = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')

        
        with tf.variable_scope('learning_rate'): 
            # global step
            self.global_step = tf.Variable(0, trainable=False)
            self.actor_decay_learning_rate = tf.train.exponential_decay(self.actor_learning_rate, self.global_step, 100000, 0.96, staircase=True)
            self.critic_decay_learning_rate = tf.train.exponential_decay(self.critic_learning_rate, self.global_step, 100000, 0.96, staircase=True)
        
        with tf.device(self.device):
            # Op for periodically updating target network with online network
            # weights with regularization
            self.generate_param_updater()
           
            self.predicted_q_value = tf.placeholder(tf.float32, [None, 1])
            # Define loss and optimization Op
            self.squared = tf.square(tf.subtract(self.predicted_q_value,self.v))
            self.l2_loss = tf.losses.get_regularization_loss(scope="net/critic")
            self.loss = tf.reduce_mean(self.squared) + self.l2_loss 
            self.critic_optimize = tf.train.AdamOptimizer(self.critic_decay_learning_rate).minimize(self.loss, global_step=self.global_step) 
            self.action_grads = tf.gradients(self.v, self.action)[0]
            self.actor_gradients = tf.gradients(self.a, self.a_params, -self.action_grads)
            self.actor_optimize = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self.actor_gradients, self.a_params), global_step=self.global_step)
            
            # inverting gradients
            self.inverting_gradients_placeholder = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='inverting_gradients')
            self._dq_da = tf.gradients(self.v, self.action)[0] # q, a 
            self._grad = tf.gradients(self.a, self.a_params, -self.inverting_gradients_placeholder)
            self._train_actor = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self._grad, self.a_params),global_step=self.global_step)
            
            


    def _build_net(self,scope):
       
        with tf.device(self.device):        
            with tf.variable_scope(scope + '/critic'):
                
                
                '''
                net = tf.layers.dense(self.inputs, LAYER_1, tf.nn.relu, name='critic_L1')
                initializer = tf.variance_scaling_initializer()
                s_union_weights = tf.Variable(initializer.__call__([LAYER_1, LAYER_2]), name='critic_L2_Ws')
                a_union_weights = tf.Variable(initializer.__call__([self.a_dim, LAYER_2]), name='critic_L2_Wa')
                union_biases = tf.Variable(tf.zeros([LAYER_2]), name='critic_L2_b')
                net = tf.nn.relu(tf.matmul(net, s_union_weights) + tf.matmul(self.action, a_union_weights) + union_biases,name='critic_L2')
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                v = tf.layers.dense(net, self.a_dim, kernel_initializer=w_init, name='critic_output')
                '''
                regularizer = tf.contrib.layers.l2_regularizer(scale=LAMBDA)
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1, weights_regularizer=regularizer, activation_fn=tf.nn.leaky_relu)
                l2_a = tf.contrib.layers.fully_connected(self.action, LAYER_2, weights_regularizer=regularizer, activation_fn=None)
                l2_s = tf.contrib.layers.fully_connected(l1, LAYER_2, weights_regularizer=regularizer,activation_fn=None)
                l2 = tf.nn.leaky_relu(l2_s + l2_a)
                v = tf.contrib.layers.fully_connected(l2, 1, weights_regularizer=regularizer, activation_fn=None)
				
            with tf.variable_scope(scope + '/actor'):
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1,  activation_fn=tf.nn.leaky_relu) # tf.nn.leaky_relu tf.nn.relu
                l2 = tf.contrib.layers.fully_connected(l1, LAYER_2,  activation_fn=tf.nn.leaky_relu)
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=tf.nn.tanh) # (para el ddpg)
                #a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=None) # (para el inverted)
                scaled_a = tf.multiply(a,self.max_action) #(para el ddpg)
                #scaled_a = a # (para el inverted)
                
                       
        saver = tf.train.Saver()
        return v, a, scaled_a, saver

    def train(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # print(L2_LOSS)
        a_outs = self.predict_action(s_batch)
        self.actor_train(s_batch, a_outs)
        
        self.update_target_network()

        return

    def test_gradient(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # train critic
        #ac_tor_grads = self._critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        #print('a grads',ac_tor_grads)
        actions = self.predict_action(s_batch)
        
        upper = self.max_action
        lower = self.min_action

        # get dq/da array, action array
        #print(upper, '***************')
        dq_das = self.sess.run([self._dq_da], feed_dict={self.inputs: s_batch, self.action:actions})[0]
        # inverting gradients, if dq_da >= 0, apply upper method, else lower method
        inverting_gradients = []
        #'''
        # print('1 dq_das, actions',dq_das, actions)
        '''
        # print('dq_das, actions',dq_das, actions)
        for dq_da, action in zip(dq_das, actions):
            # print('dq_da, action',dq_da, action)
            if dq_da >= 0.0:
                inverting_gradients.append(dq_da * (self.max_action - action) / (self.max_action - self.min_action))
            else:
                inverting_gradients.append(dq_da * (action - self.min_action) / (self.max_action - self.min_action))
        inverting_gradients = np.array(inverting_gradients).reshape(-1, 1)
		'''

        for i in range(MINIBATCH_SIZE):
            #print('2', i,dq_das[i])
            for j in range(self.a_dim):
                if dq_das[i][j] >= 0.0:
                    dq_das[i][j] = dq_das[i][j] * (self.max_action - actions[i][j]) / (self.max_action - self.min_action)
                else:
                    dq_das[i][j] = dq_das[i][j] * (actions[i][j] - self.min_action) / (self.max_action - self.min_action)
        
        # print(dq_das,inverting_gradients)
        # exit()
        inverting_gradients = dq_das 
        
        # print('2 dq_das, actions',dq_das, actions)
        
         
        #print('1','inverting_gradients',inverting_gradients)
        
        # print('2','inverting_gradients',inverting_gradients,dq_das, actions)
        # time.sleep(1)
        # update actor
        self.sess.run(self._train_actor, feed_dict={self.inputs: s_batch, self.inverting_gradients_placeholder: inverting_gradients})
        self.update_target_network()
        return

    
    def _critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.action_grads], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def update_target_network(self):
        self.sess.run([self.a_updater,self.c_updater])

    def generate_param_updater(self):
        self.a_updater = [self.a_params_target[i].assign(tf.multiply(self.a_params[i], self.tau) + tf.multiply(self.a_params_target[i], 1. - self.tau))
                for i in range(len(self.a_params))]
        self.c_updater = [self.c_params_target[i].assign(tf.multiply(self.c_params[i], self.tau) + tf.multiply(self.c_params_target[i], 1. - self.tau))
                for i in range(len(self.c_params))]

    def critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.loss,self.critic_optimize], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def actor_train(self,inputs, action):
        return self.sess.run(self.actor_optimize, feed_dict={
            self.inputs: inputs,
            self.action: action
        })

    def save(self):
        self.saver.save(self.sess,"./model/model.ckpt")
        self.saver_target.save(self.sess,"./model/model_target.ckpt")
        print("Model saved in file: actor_model")

    
    def load(self):
        self.saver.restore(self.sess,"./model/model.ckpt")
        self.saver_target.restore(self.sess,"./model/model_target.ckpt")
        


    def critic_predict_target(self, state, action):
        return self.sess.run(self.v_target, feed_dict={
            self.inputs: state,
            self.action: action
        })
        
    def predict_action_target(self, state):
        return self.sess.run(self.scaled_a_target, feed_dict={
            self.inputs: state
        })

    def predict_action(self, state):
        return self.sess.run(self.scaled_a, feed_dict={
            self.inputs: state
        })



def normalizing_state(state):
    #state = [V,P] being V_min = 0, V_max = 210, P_min = 0, P_max = 25000 we use ((xi-x_min)/(x_max-x_min))-1
    V_min = 0
    V_max = 210
    P_min = 0
    P_max = 54000
    DeltaP_min = -15000
    DeltaP_max = 15000

    st = [(2*(state[0]-V_min)/(V_max-V_min))-1, (2*(state[1]-P_min)/(P_max-P_min))-1,(2*(state[1]-DeltaP_min)/(DeltaP_max-DeltaP_min))-1]

    return st


if __name__ == '__main__':
    from replay_buffer import ReplayBuffer
    import gym
    import gym_mppt
    ACTOR_LEARNING_RATE = 0.0001
    CRITIC_LEARNING_RATE =  0.001
    # Soft target update param
    TAU = 0.001
    DEVICE = '/cpu:0'
    # ENV_NAME = 'MountainCarContinuous-v0'
    ENV_NAME = 'mppt_shaded-v0'#'Pendulum-v0'
    # import gym_foo
    # ENV_NAME = 'nessie_end_to_end-v0'
    max_action = 10.
    min_action = -10.
    epochs = 10000
    epsilon = 1.0
    min_epsilon = 0.1
    EXPLORE = 15000
    BUFFER_SIZE = 50000
    RANDOM_SEED = 51234
    MINIBATCH_SIZE = 64# 32 # 5
    with tf.Session() as sess:
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        env = gym.make(ENV_NAME)
        state_dim = np.size(env.reset()) #env.observation_space.shape[0]
        action_dim = 1 #env.action_space.shape[0]
        ddpg = DDPG(sess, state_dim, action_dim, max_action, min_action, ACTOR_LEARNING_RATE, CRITIC_LEARNING_RATE, TAU, RANDOM_SEED,device=DEVICE)
        sess.run(tf.global_variables_initializer())
        #ddpg.load()
        replay_buffer = ReplayBuffer(BUFFER_SIZE, RANDOM_SEED)
        ruido = OUNoise(action_dim, mu = 0.0)
        Reward_episodios = []
        llegadas =0
        for i in range(epochs):
            state = env.reset()
            normalized_state = normalizing_state(state)
            #print('EL ESTADO RESETEADO ES', state, state.shape)
            #print('epoch = ', i)
            done = False
            epsilon -= (epsilon/EXPLORE)
            epsilon = np.maximum(min_epsilon,epsilon)
            episode_r = 0.
            step = 0 
            max_steps = 45 #es solo a efecto de contador
            r_episodio_actual = []
            #llegadas =0
            while (not done):
                #print('step =', step)
                #wait = input("PRESS ENTER TO CONTINUE.")
                action = ddpg.predict_action(np.reshape(normalized_state,(1,state_dim)))
                action1 = action
                #print('LA ACCION sin clipear ES', action1, action1.shape) 
                action = np.clip(action1,min_action,max_action)
                #action = action + max(epsilon,0)*ruido.noise()
                #e-greedy action selection
                if random.random() < epsilon:
                    action = random.uniform(min_action,max_action)
                    #print('Explorando')

                action = np.clip(action,min_action,max_action)
                #print('ruido =', max(epsilon,0)*ruido.noise(),'epsilon =',epsilon)
                #print('LA ACCION clipeada ES', action, action.shape)
                
                next_state, reward, done, info = env.step(action)
                normalized_next_state = normalizing_state(next_state)
                #print('EL NEXT_ESTADO ES', next_state, next_state.shape) 
                #reward = np.clip(reward,-1.,1.)
                #print('instaneous r = ',reward)
                #print('Steps = ', info['Steps'],'V0 =',info['v0'])
                #print('normalized_state =',normalized_state)
                #wait = input("PRESS ENTER TO CONTINUE.")
                replay_buffer.add(np.reshape(normalized_state, (state_dim,)), np.reshape(action, (action_dim,)), reward,
                                      done, np.reshape(normalized_next_state, (state_dim,)))
                state = next_state
                episode_r = episode_r + reward
                normalized_state = normalizing_state(state)

                r_episodio_actual.append(reward)
                if replay_buffer.size() > MINIBATCH_SIZE:
                    s_batch, a_batch, r_batch, t_batch, s2_batch = replay_buffer.sample_batch(MINIBATCH_SIZE)
                    # train ddpg normally:
                    ddpg.train(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
                    #train with inverted gradients
                    #ddpg.test_gradient(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
                #print(i, step, 'last r', round(reward,3), 'episode reward',round(episode_r,3), 'epsilon', round(epsilon,3))
                #print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state,'last r =', round(reward[0][0],3), 'episode reward =',round(episode_r[0][0],3), 'epsilon =', round(epsilon,3))
                step += 1

            #if info['Steps'] > max_steps:
             #   llegadas +=1
            #print ('--------------------------------------------')
            #print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state, 'accion =',action,'last r =', reward, 'episode reward =',episode_r, 'epsilon =', round(epsilon,3))
            #print ('--------------------------------------------')

            print(i, info['Temperatura'], info['Irradiancia'], info['Shading'])

            Reward_episodios.append(r_episodio_actual)
            np.save('Reward_episodios_DDPG01.npy',Reward_episodios)
            

            print('FIN epoch = ',i)                


        ddpg.save()

0 30 969 [7, 10, 5, 10, 5, 10]
FIN epoch =  0
1 50 701 [7, 10, 7, 10, 7, 10]
FIN epoch =  1
2 16 148 [2, 10, 1, 10, 1, 10]
FIN epoch =  2
3 30 210 [7, 10, 7, 10, 5, 10]
FIN epoch =  3
4 17 611 [6, 10, 2, 10, 8, 10]
FIN epoch =  4
5 16 652 [10, 10, 7, 10, 10, 10]
FIN epoch =  5
6 24 161 [3, 10, 5, 10, 7, 10]
FIN epoch =  6
7 34 537 [5, 10, 5, 10, 2, 10]
FIN epoch =  7
8 50 978 [5, 10, 10, 10, 4, 10]
FIN epoch =  8
9 15 435 [9, 10, 6, 10, 8, 10]
FIN epoch =  9
10 24 674 [1, 10, 10, 10, 7, 10]
FIN epoch =  10
11 23 793 [7, 10, 6, 10, 9, 10]
FIN epoch =  11
12 39 515 [2, 10, 8, 10, 9, 10]
FIN epoch =  12
13 13 568 [10, 10, 9, 10, 2, 10]
FIN epoch =  13
14 10 449 [2, 10, 9, 10, 1, 10]
FIN epoch =  14
15 32 352 [2, 10, 9, 10, 9, 10]
FIN epoch =  15
16 21 595 [10, 10, 4, 10, 4, 10]
FIN epoch =  16
17 26 249 [6, 10, 9, 10, 6, 10]
FIN epoch =  17
18 29 995 [4, 10, 4, 10, 4, 10]
FIN epoch =  18
19 30 913 [8, 10, 3, 10, 7, 10]
FIN epoch =  19
20 24 369 [7, 10, 5, 10, 4, 10]
FIN epoch =  20
21 11 

c:\users\archi\python code\btp\ddpg_paper2\mppt-gym-rama2_mdp\gym_mppt\envs\pvmodel_shaded.py:37: RuntimeWarning: overflow encountered in exp
  I = self.Np * Iph - self.Np * Irs * (np.exp(self.q / (self.k * Tcell * self.A) * vx * (1/SH) / self.Ns) - 1)
c:\users\archi\python code\btp\ddpg_paper2\mppt-gym-rama2_mdp\gym_mppt\envs\pvmodel_shaded.py:52: RuntimeWarning: invalid value encountered in subtract
  In = I - fv/fdv
c:\users\archi\python code\btp\ddpg_paper2\mppt-gym-rama2_mdp\gym_mppt\envs\pvmodel_shaded.py:43: RuntimeWarning: invalid value encountered in greater
  while diff > 1e-3:
c:\users\archi\python code\btp\ddpg_paper2\mppt-gym-rama2_mdp\gym_mppt\envs\shaded.py:25: RuntimeWarning: invalid value encountered in greater
  Ipv.append(max(0, IUN_i, ISH_i))


152 18 555 [1, 10, 2, 10, 7, 10]
FIN epoch =  152
153 39 401 [10, 10, 9, 10, 5, 10]
FIN epoch =  153
154 34 706 [2, 10, 4, 10, 2, 10]
FIN epoch =  154
155 44 920 [2, 10, 9, 10, 9, 10]
FIN epoch =  155
156 39 195 [5, 10, 10, 10, 5, 10]
FIN epoch =  156
157 44 242 [6, 10, 5, 10, 2, 10]
FIN epoch =  157
158 42 173 [6, 10, 4, 10, 5, 10]
FIN epoch =  158
159 11 993 [2, 10, 4, 10, 6, 10]
FIN epoch =  159
160 28 516 [3, 10, 2, 10, 1, 10]
FIN epoch =  160
161 49 506 [3, 10, 7, 10, 8, 10]
FIN epoch =  161
162 14 812 [9, 10, 8, 10, 7, 10]
FIN epoch =  162
163 24 522 [10, 10, 3, 10, 2, 10]
FIN epoch =  163
164 38 916 [5, 10, 7, 10, 10, 10]
FIN epoch =  164
165 12 951 [8, 10, 4, 10, 3, 10]
FIN epoch =  165
166 45 156 [2, 10, 4, 10, 6, 10]
FIN epoch =  166
167 49 313 [4, 10, 5, 10, 1, 10]
FIN epoch =  167
168 21 297 [9, 10, 5, 10, 1, 10]
FIN epoch =  168
169 47 193 [1, 10, 9, 10, 8, 10]
FIN epoch =  169
170 11 784 [6, 10, 6, 10, 7, 10]
FIN epoch =  170
171 44 472 [8, 10, 9, 10, 4, 10]
FIN epoch =  

315 18 850 [4, 10, 6, 10, 1, 10]
FIN epoch =  315
316 37 359 [3, 10, 10, 10, 4, 10]
FIN epoch =  316
317 38 983 [9, 10, 4, 10, 3, 10]
FIN epoch =  317
318 18 102 [10, 10, 10, 10, 1, 10]
FIN epoch =  318
319 50 765 [2, 10, 5, 10, 2, 10]
FIN epoch =  319
320 48 259 [5, 10, 4, 10, 8, 10]
FIN epoch =  320
321 20 940 [10, 10, 1, 10, 6, 10]
FIN epoch =  321
322 50 170 [10, 10, 2, 10, 10, 10]
FIN epoch =  322
323 41 394 [1, 10, 9, 10, 4, 10]
FIN epoch =  323
324 36 645 [3, 10, 1, 10, 9, 10]
FIN epoch =  324
325 11 833 [7, 10, 4, 10, 5, 10]
FIN epoch =  325
326 45 394 [10, 10, 7, 10, 2, 10]
FIN epoch =  326
327 42 927 [7, 10, 9, 10, 2, 10]
FIN epoch =  327
328 13 144 [2, 10, 1, 10, 4, 10]
FIN epoch =  328
329 40 728 [1, 10, 6, 10, 7, 10]
FIN epoch =  329
330 47 611 [1, 10, 4, 10, 4, 10]
FIN epoch =  330
331 10 979 [4, 10, 4, 10, 5, 10]
FIN epoch =  331
332 20 592 [1, 10, 8, 10, 8, 10]
FIN epoch =  332
333 35 217 [9, 10, 10, 10, 9, 10]
FIN epoch =  333
334 31 939 [8, 10, 6, 10, 5, 10]
FIN epoch

478 12 627 [9, 10, 5, 10, 8, 10]
FIN epoch =  478
479 38 455 [7, 10, 7, 10, 7, 10]
FIN epoch =  479
480 20 950 [10, 10, 6, 10, 7, 10]
FIN epoch =  480
481 28 642 [7, 10, 8, 10, 10, 10]
FIN epoch =  481
482 35 823 [1, 10, 3, 10, 2, 10]
FIN epoch =  482
483 45 121 [2, 10, 6, 10, 5, 10]
FIN epoch =  483
484 24 469 [2, 10, 6, 10, 1, 10]
FIN epoch =  484
485 19 535 [1, 10, 6, 10, 9, 10]
FIN epoch =  485
486 32 998 [9, 10, 5, 10, 8, 10]
FIN epoch =  486
487 28 643 [2, 10, 1, 10, 8, 10]
FIN epoch =  487
488 49 151 [2, 10, 9, 10, 3, 10]
FIN epoch =  488
489 11 201 [5, 10, 4, 10, 5, 10]
FIN epoch =  489
490 29 202 [10, 10, 5, 10, 9, 10]
FIN epoch =  490
491 37 419 [7, 10, 5, 10, 9, 10]
FIN epoch =  491
492 43 497 [7, 10, 3, 10, 5, 10]
FIN epoch =  492
493 12 524 [7, 10, 4, 10, 2, 10]
FIN epoch =  493
494 39 954 [5, 10, 8, 10, 3, 10]
FIN epoch =  494
495 38 851 [6, 10, 2, 10, 6, 10]
FIN epoch =  495
496 18 641 [4, 10, 2, 10, 8, 10]
FIN epoch =  496
497 16 524 [1, 10, 3, 10, 2, 10]
FIN epoch =  4

...

FIN epoch =  9288
9289 16 816 [3, 10, 7, 10, 8, 10]
FIN epoch =  9289
9290 16 695 [8, 10, 4, 10, 1, 10]
FIN epoch =  9290
9291 43 312 [3, 10, 10, 10, 6, 10]
FIN epoch =  9291
9292 34 826 [8, 10, 6, 10, 6, 10]
FIN epoch =  9292
9293 37 673 [10, 10, 6, 10, 6, 10]
FIN epoch =  9293
9294 22 704 [4, 10, 3, 10, 7, 10]
FIN epoch =  9294
9295 37 749 [4, 10, 6, 10, 10, 10]
FIN epoch =  9295
9296 27 128 [6, 10, 1, 10, 9, 10]
FIN epoch =  9296
9297 19 530 [5, 10, 2, 10, 8, 10]
FIN epoch =  9297
9298 15 394 [5, 10, 4, 10, 4, 10]
FIN epoch =  9298
9299 18 866 [10, 10, 2, 10, 6, 10]
FIN epoch =  9299
9300 15 871 [1, 10, 3, 10, 2, 10]
FIN epoch =  9300
9301 36 433 [1, 10, 10, 10, 9, 10]
FIN epoch =  9301
9302 30 351 [1, 10, 4, 10, 2, 10]
FIN epoch =  9302
9303 38 734 [5, 10, 5, 10, 8, 10]
FIN epoch =  9303
9304 35 737 [3, 10, 2, 10, 1, 10]
FIN epoch =  9304
9305 19 795 [2, 10, 5, 10, 6, 10]
FIN epoch =  9305
9306 23 118 [9, 10, 7, 10, 10, 10]
FIN epoch =  9306
9307 10 612 [4, 10, 5, 10, 10, 10]
FIN e

FIN epoch =  9445
9446 45 751 [10, 10, 3, 10, 3, 10]
FIN epoch =  9446
9447 23 696 [6, 10, 3, 10, 7, 10]
FIN epoch =  9447
9448 17 805 [2, 10, 6, 10, 1, 10]
FIN epoch =  9448
9449 19 193 [5, 10, 8, 10, 6, 10]
FIN epoch =  9449
9450 40 469 [7, 10, 6, 10, 9, 10]
FIN epoch =  9450
9451 49 371 [1, 10, 7, 10, 2, 10]
FIN epoch =  9451
9452 49 292 [8, 10, 1, 10, 8, 10]
FIN epoch =  9452
9453 41 866 [7, 10, 7, 10, 1, 10]
FIN epoch =  9453
9454 40 248 [2, 10, 10, 10, 2, 10]
FIN epoch =  9454
9455 14 128 [5, 10, 2, 10, 2, 10]
FIN epoch =  9455
9456 50 834 [4, 10, 1, 10, 2, 10]
FIN epoch =  9456
9457 46 699 [10, 10, 10, 10, 5, 10]
FIN epoch =  9457
9458 30 211 [5, 10, 2, 10, 9, 10]
FIN epoch =  9458
9459 49 853 [5, 10, 2, 10, 7, 10]
FIN epoch =  9459
9460 28 954 [7, 10, 9, 10, 3, 10]
FIN epoch =  9460
9461 32 117 [9, 10, 6, 10, 8, 10]
FIN epoch =  9461
9462 45 633 [9, 10, 3, 10, 4, 10]
FIN epoch =  9462
9463 19 404 [6, 10, 8, 10, 1, 10]
FIN epoch =  9463
9464 24 989 [9, 10, 5, 10, 6, 10]
FIN epoc

FIN epoch =  9602
9603 14 104 [3, 10, 8, 10, 1, 10]
FIN epoch =  9603
9604 31 665 [7, 10, 9, 10, 9, 10]
FIN epoch =  9604
9605 35 384 [6, 10, 3, 10, 2, 10]
FIN epoch =  9605
9606 47 309 [4, 10, 5, 10, 6, 10]
FIN epoch =  9606
9607 21 273 [6, 10, 8, 10, 8, 10]
FIN epoch =  9607
9608 34 371 [6, 10, 1, 10, 7, 10]
FIN epoch =  9608
9609 24 388 [8, 10, 3, 10, 3, 10]
FIN epoch =  9609
9610 27 354 [10, 10, 10, 10, 5, 10]
FIN epoch =  9610
9611 35 622 [10, 10, 9, 10, 7, 10]
FIN epoch =  9611
9612 36 676 [1, 10, 5, 10, 3, 10]
FIN epoch =  9612
9613 30 981 [6, 10, 4, 10, 3, 10]
FIN epoch =  9613
9614 22 317 [6, 10, 4, 10, 6, 10]
FIN epoch =  9614
9615 38 730 [10, 10, 1, 10, 6, 10]
FIN epoch =  9615
9616 42 198 [9, 10, 5, 10, 5, 10]
FIN epoch =  9616
9617 28 333 [3, 10, 9, 10, 5, 10]
FIN epoch =  9617
9618 24 513 [7, 10, 10, 10, 3, 10]
FIN epoch =  9618
9619 15 785 [1, 10, 1, 10, 6, 10]
FIN epoch =  9619
9620 36 974 [9, 10, 8, 10, 5, 10]
FIN epoch =  9620
9621 12 375 [5, 10, 1, 10, 6, 10]
FIN epo

FIN epoch =  9759
9760 17 124 [9, 10, 3, 10, 6, 10]
FIN epoch =  9760
9761 50 894 [7, 10, 6, 10, 5, 10]
FIN epoch =  9761
9762 10 179 [4, 10, 1, 10, 4, 10]
FIN epoch =  9762
9763 41 961 [6, 10, 8, 10, 9, 10]
FIN epoch =  9763
9764 44 534 [5, 10, 8, 10, 4, 10]
FIN epoch =  9764
9765 26 854 [10, 10, 10, 10, 5, 10]
FIN epoch =  9765
9766 34 136 [10, 10, 7, 10, 1, 10]
FIN epoch =  9766
9767 30 708 [1, 10, 10, 10, 5, 10]
FIN epoch =  9767
9768 11 802 [4, 10, 2, 10, 9, 10]
FIN epoch =  9768
9769 27 901 [4, 10, 8, 10, 10, 10]
FIN epoch =  9769
9770 32 281 [2, 10, 3, 10, 1, 10]
FIN epoch =  9770
9771 10 699 [5, 10, 5, 10, 3, 10]
FIN epoch =  9771
9772 16 249 [5, 10, 8, 10, 1, 10]
FIN epoch =  9772
9773 47 872 [7, 10, 2, 10, 4, 10]
FIN epoch =  9773
9774 29 146 [10, 10, 2, 10, 8, 10]
FIN epoch =  9774
9775 36 873 [5, 10, 7, 10, 8, 10]
FIN epoch =  9775
9776 29 943 [6, 10, 4, 10, 6, 10]
FIN epoch =  9776
9777 22 831 [1, 10, 2, 10, 4, 10]
FIN epoch =  9777
9778 21 150 [10, 10, 2, 10, 9, 10]
FIN e

FIN epoch =  9916
9917 38 262 [1, 10, 6, 10, 6, 10]
FIN epoch =  9917
9918 33 163 [6, 10, 2, 10, 8, 10]
FIN epoch =  9918
9919 22 258 [5, 10, 7, 10, 8, 10]
FIN epoch =  9919
9920 49 962 [7, 10, 10, 10, 3, 10]
FIN epoch =  9920
9921 43 741 [3, 10, 6, 10, 9, 10]
FIN epoch =  9921
9922 16 149 [1, 10, 6, 10, 3, 10]
FIN epoch =  9922
9923 35 316 [9, 10, 10, 10, 10, 10]
FIN epoch =  9923
9924 33 734 [7, 10, 8, 10, 2, 10]
FIN epoch =  9924
9925 39 786 [2, 10, 9, 10, 4, 10]
FIN epoch =  9925
9926 38 982 [8, 10, 6, 10, 3, 10]
FIN epoch =  9926
9927 13 853 [9, 10, 2, 10, 6, 10]
FIN epoch =  9927
9928 16 696 [2, 10, 9, 10, 8, 10]
FIN epoch =  9928
9929 14 814 [10, 10, 3, 10, 4, 10]
FIN epoch =  9929
9930 37 357 [10, 10, 4, 10, 5, 10]
FIN epoch =  9930
9931 39 595 [10, 10, 6, 10, 7, 10]
FIN epoch =  9931
9932 15 706 [10, 10, 8, 10, 2, 10]
FIN epoch =  9932
9933 16 732 [4, 10, 2, 10, 5, 10]
FIN epoch =  9933
9934 31 987 [9, 10, 3, 10, 2, 10]
FIN epoch =  9934
9935 45 902 [6, 10, 7, 10, 6, 10]
FIN e